In [0]:
# ============================================================
# SENTINEL COMMERCE
# Notebook: 02_bronze_orders_autoloader
#
# Layer   : Bronze
# Pattern : Incremental file ingestion using Auto Loader
# Source  : Unity Catalog Volume
# Target  : Delta table
# ============================================================

### ===========================================================
# KEY POINTS
# For JSON/CSV/XML, Auto Loader defaults to inferring columns as strings rather than aggressively guessing their data types.
# availableNow=True helps to process all files in the source directory at once.
# The _rescued_data column is automatically created by Auto Loader to capture any data that cannot be parsed.
# The _metadata column is automatically created by Auto Loader to capture file metadata.
# The _commit_version column is automatically created by Auto Loader to capture the version of the source directory when the file was processed.
# For Auto Loader with schema inference, addNewColumns is the default schema-evolution mode
# Schema hints tell Auto Loader how columns should be read; when incoming data doesn't match the hinted type, that data can be rescued rather than simply cast away.
# Changing transformation code does not make Auto Loader forget what it has already processed. The checkpoint persists independently of your new code.

# ===========================================================


# BRONZE
# ────────────
# Flexible
# Source-oriented
# Preserve data
# Allow evolution


# SILVER
# ────────────
# Contract-driven
# Typed
# Validated
# Deduplicated


# GOLD
# ────────────
# Stable
# Business-defined
# Consumer-facing
from pyspark.sql.functions import *
from pyspark.sql.types import *

SOURCE_PATH = "/Volumes/sentinel_dev/landing/source_files/orders"
SCHEMA_PATH = "/Volumes/sentinel_dev/landing/source_files/_schemas/orders"
CHECKPOINT_PATH = "/Volumes/sentinel_dev/landing/source_files/_checkpoints/orders"
TARGET_TABLE = "sentinel_dev.bronze.orders_raw"

RESCUE_SCHEMA_PATH = (
    "/Volumes/sentinel_dev/landing/source_files/"
    "_schemas/orders_rescue_test"
)

RESCUE_CHECKPOINT_PATH = (
    "/Volumes/sentinel_dev/landing/source_files/"
    "_checkpoints/orders_rescue_test"
)

RESCUE_TARGET_TABLE = (
    "sentinel_dev.bronze.orders_rescue_test"
)

In [0]:
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("rescuedDataColumn", "_rescued_data")
        .option(
            "cloudFiles.schemaHints",
            """
            quantity INT,
            unit_price DOUBLE,
            total_amount DOUBLE
            """
        )
        .load(SOURCE_PATH)
)

In [0]:
orders_stream.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_stream = (
    orders_stream
        .selectExpr(
            "*",
            "_metadata.file_path AS source_file",
            "_metadata.file_name AS source_file_name",
            "_metadata.file_modification_time AS source_file_modified_at"
        )
        .withColumn(
            "ingested_at",
            current_timestamp()
        )
)

In [0]:
bronze_stream.printSchema()

In [0]:
bronze_query = (
    bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)

bronze_query.awaitTermination()

print("Bronze ingestion completed.")

In [0]:
bronze_count = spark.table(TARGET_TABLE).count()

print(f"Bronze record count: {bronze_count:,}")

In [0]:
display(
    spark.table(TARGET_TABLE)
        .filter("_rescued_data IS NOT NULL")
        .select(
            "order_id",
            "quantity",
            "unit_price",
            "_rescued_data",
            "source_file_name",
            "ingested_at"
        )
)

In [0]:
spark.sql("""DESCRIBE DETAIL sentinel_dev.bronze.orders_raw""").display()

In [0]:
print(
    spark.table(TARGET_TABLE).count()
)

In [0]:
#Rescue data

rescue_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", RESCUE_SCHEMA_PATH)
        .option("rescuedDataColumn", "_rescued_data")
        .option(
            "cloudFiles.schemaHints",
            """
            quantity INT,
            unit_price DOUBLE,
            total_amount DOUBLE
            """
        )
        .load(SOURCE_PATH)
)

In [0]:
rescue_stream.printSchema()

In [0]:
rescue_query = (
    rescue_stream.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            RESCUE_CHECKPOINT_PATH
        )
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(RESCUE_TARGET_TABLE)
)

rescue_query.awaitTermination()

In [0]:
spark.table(RESCUE_TARGET_TABLE).printSchema()

In [0]:
display(
    spark.table(RESCUE_TARGET_TABLE)
        .filter("_rescued_data IS NOT NULL")
        .select(
            "order_id",
            "quantity",
            "unit_price",
            "_rescued_data",
            "source_file_name",
            "ingested_at"
        )
)

In [0]:
from pyspark.sql.functions import (
    count,
    sum as spark_sum,
    when,
    max as spark_max
)

bronze_df = spark.table(RESCUE_TARGET_TABLE)

bronze_metrics = (
    bronze_df
        .agg(
            count("*").alias("total_records"),

            spark_sum(
                when(
                    bronze_df["_rescued_data"].isNotNull(),
                    1
                ).otherwise(0)
            ).alias("rescued_records"),

            spark_max(
                "ingested_at"
            ).alias("latest_ingestion")
        )
)

display(bronze_metrics)